# NSF Future Manufacturing Data Challenge — The Regularizers

**Vedangi Bengali, Dhawal Chaudhari — Texas A&M University**

Predicting spatially varying laser-track geometry from in-situ thermal imaging.

This notebook runs the complete submitted workflow end to end: it loads the raw
data, extracts the ground-truth geometry from the profilometer height maps,
builds the thermal and masked-SEM features, trains the quantile models, runs
every validation, writes the machine-readable prediction file, and reproduces
the report figures.

All modelling logic lives in `run_pipeline.py` so that the notebook and the
command-line entry point cannot diverge. Nothing is re-implemented here.

**Runtime** ~8 minutes on an Apple-silicon laptop (CPU only, no GPU).
**Seeds** every model uses `random_state=0`; the pipeline has no other
stochastic step, so `metrics.json` reproduces byte-identically.


## 0. Environment and data layout

Install requirements and place the Zenodo data as shown. This cell only prints
the expected layout — see `README.txt` for the download commands.

```
data/raw/thermal/Thermal_{8,10,14,21}.mat
data/raw/height_maps/Heightmap_{8,10,14,21}.ASC
data/raw/sem/SEM_{8,10,14,21}/Scale_*.tif
```


In [ ]:
import sys, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd()
DATA = REPO / "data" / "raw"
OUT  = REPO / "outputs"
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "src"))

for sub in ("thermal", "height_maps", "sem"):
    n = len(list((DATA / sub).glob("*"))) if (DATA / sub).exists() else 0
    print(f"{sub:12s} {n} entries")
print("python", sys.version.split()[0])
for m in ("numpy", "scipy", "pandas", "sklearn", "matplotlib", "PIL", "h5py"):
    mod = __import__(m); print(f"  {m:12s} {getattr(mod,'__version__','?')}")

## 1. Ground-truth geometry extraction

These are bead-on-plate **remelt** tracks: no powder was fed, so the crown rises
only a few microns and a height-threshold rule fails (it finds a bead in 6 of
400 bins on track 8). We segment on **surface finish** instead — the
resolidified pool is optically smooth and densely measured, the rough substrate
is neither. Three independent signals (measurement validity, local roughness,
residual height dome) agree to within a few pixels.

In [ ]:
import run_pipeline as rp

targets = rp.build_targets(DATA / "height_maps", 8, demo_out=OUT)
v = targets[targets["valid"]]
print(f"track 8: {len(v)}/{len(targets)} usable 0.2 mm bins")
print(f"  width  mean {v['width_mm'].mean():.3f} mm   sd {v['width_mm'].std():.3f} mm")
print(f"  crown rise median {1000*v['rise_mm'].median():.1f} um")
print("  split-half reliability:", targets.attrs["reliability"])

In [ ]:
from IPython.display import Image, display
display(Image(str(OUT / "ground_truth_extraction.png")))

## 2. Thermal descriptors and masked SEM features

Eleven interpretable melt-pool descriptors per frame plus rolling statistics.
The SEM tiles image the region *containing* the finished track, so the track is
located and masked out with a 0.30 mm margin before any feature is computed.

In [ ]:
feat = rp.build_features(DATA / "thermal", 8)
sem, band_mm, n_tiles = rp.build_sem_features(DATA / "sem", 8,
                                              band_out=True, demo_out=OUT)
print(f"thermal features: {feat.shape[1]-2} columns x {len(feat)} bins")
print(f"SEM: {n_tiles} tiles, visible track band {band_mm:.3f} mm, "
      f"{sem.shape[1]-2} substrate features")
display(Image(str(OUT / "sem_masking.png")))

## 3. Full pipeline

Runs every stage: ground truth for all four tracks, thermal and SEM features,
the three-way modality ablation, leave-one-track-out validation, within-track
blocked spatial CV, the 21-combination local-skill search, the model bake-off,
the thermal-registration check, conformal calibration, the final held-out test
on track 21, the machine-readable prediction file, and all figures.

`--headline thermal` selects which configuration the figures use; it does not
change any metric. The primary *unbiased* result is chosen by the selection
protocol (lowest LOTO CRPS) and recorded in `metrics.json` regardless.

In [ ]:
r = subprocess.run([sys.executable, "run_pipeline.py",
                    "--data-root", str(DATA), "--headline", "thermal"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout[-3500:])
print("exit code:", r.returncode)

## 4. Selection protocol — how track 21 was used

Track 21 is the held-out test, so it may be scored once. Choosing a feature set
or interval type by comparing track-21 scores would turn it into a validation
set. The rule below uses the leave-one-track-out folds **only**.

In [ ]:
M  = json.load(open(OUT / "metrics.json"))
sp = M["selection_protocol"]
print("rule:", sp["rule"], "\n")
print("selected feature set :", sp["selected_feature_set"])
print("each LOTO rule picks :", sp["what_each_LOTO_rule_would_pick"])
print("stable across rules  :", sp["selection_is_stable_across_rules"])
p = sp["PRIMARY_unbiased_final_track21"]
print(f"\nPRIMARY (unbiased) track 21: MAE {p['MAE_mm']:.4f} mm | "
      f"CRPS {p['CRPS_mm']:.4f} | 80% coverage {p['coverage_80pct_interval']:.3f}")

## 5. Results

In [ ]:
rows = []
for k, v in M.items():
    if k.startswith(("LOTO_holdout", "WITHIN_TRACK", "FINAL")) and not k.endswith("_CQR"):
        rows.append(dict(evaluation=k, n=v["n_points"], MAE_mm=v["MAE_mm"],
                         CRPS_mm=v["CRPS_mm"], cov80=v["coverage_80pct_interval"],
                         cal_err=v["calibration_error"],
                         baseline_mm=v["baseline_trainmean_MAE_mm"], r=v["pearson_r"]))
pd.DataFrame(rows).round(4)

In [ ]:
for f in ("prediction_track_21.png", "local_skill_search.png", "ablation.png"):
    display(Image(str(OUT / f)))

## 6. Why the null result on local variation is robust

The model captures the track-level geometry scale but not local variation. Four
alternative explanations were eliminated by direct measurement rather than
assertion.

In [ ]:
rel = {t: M["data_summary"][t]["reliability"] for t in ["8","10","14","21"]}
print("1. Is the reference too noisy?")
for t, v in rel.items():
    print(f"   track {t}: reliability {v['full_length_reliability']:.3f} -> "
          f"ceiling |r| {v['max_achievable_abs_r']:.3f}, noise "
          f"{100*v['noise_share_of_variance']:.0f}% of variance")

lag = M["thermal_lag_check"]
print("\n2. Is the thermal axis misregistered?")
print("  ", lag["null_baseline"]["surrogate_method"],
      "-> chance max |r| =", lag["null_baseline"]["null_mean_max_abs_r"])
print("   optimal-lag spread:", lag["optimal_lag_spread_bins"], "bins")
print("  ", lag["transfer_test_verdict"])

lss = M["local_skill_search"]
best = max(lss.items(), key=lambda kv: kv[1]["mean_skill"])
print(f"\n3. Wrong target or scale?  {len(lss)} combinations, all negative.")
print(f"   best: {best[0]} at skill {best[1]['mean_skill']:+.4f}")

print("\n4. Wrong model class?")
for k, v in M["model_bakeoff"].items():
    print(f"   {k:20s} LOTO {v['LOTO_mean_MAE_mm']:.4f}  FINAL {v['FINAL_MAE_mm']:.4f}")

## 7. Machine-readable prediction file

`TheRegularizers_Predictions.csv` holds the submitted geometry representation:
nine quantiles for each of three descriptors (left boundary, right boundary,
width) at every 0.2 mm bin of all four tracks, alongside our profilometer-derived
reference. Units are millimetres throughout; `x_mm` is position along the scan
direction in actual part coordinates.

In [ ]:
pred = pd.read_csv(OUT / "TheRegularizers_Predictions.csv")
print(pred.shape, "\ndescriptors:", sorted(pred.descriptor.unique()),
      "\nsplits:", sorted(pred.split.unique()))
print(json.dumps(M["prediction_file"], indent=2))
pred.head()

In [ ]:
# reconstructed contour for the held-out track, from the submitted file
d = pred[(pred.track_id == 21) & (pred.split == "FINAL_held_out")]
L = d[d.descriptor == "left"].sort_values("x_mm")
R = d[d.descriptor == "right"].sort_values("x_mm")
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.fill_between(L.x_mm, L.pred_q10_mm, L.pred_q90_mm, alpha=.3, color="tab:blue")
ax.fill_between(R.x_mm, R.pred_q10_mm, R.pred_q90_mm, alpha=.3, color="tab:red")
ax.plot(L.x_mm, L.pred_q50_mm, lw=1.2, color="tab:blue", label="predicted left boundary")
ax.plot(R.x_mm, R.pred_q50_mm, lw=1.2, color="tab:red",  label="predicted right boundary")
ax.plot(L.x_mm, L.reference_mm, ".", ms=2, color="k", alpha=.5, label="measured boundaries")
ax.plot(R.x_mm, R.reference_mm, ".", ms=2, color="k", alpha=.5)
ax.set_xlabel("position along track, x (mm)"); ax.set_ylabel("cross-track y (mm)")
ax.set_title("Track 21 (held out): predicted track contour with 80% bands")
ax.legend(fontsize=8); ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## 8. Conclusion

In-situ melt-pool imaging predicts the geometry scale of a completely unseen
process condition with calibrated uncertainty, and supplies an estimate at every
location — including those where post-process metrology returned nothing. It
does not resolve local variation, and that null is robust across 21 descriptor ×
scale combinations, a measurement ceiling of |r| ≈ 0.97 that is left
unexploited, an autocorrelation-matched registration null, three model classes,
and a direct trailing-edge thermal measurement.

The binding limitation is dataset size: with four tracks at four conditions,
between-condition variance rests on two points per fold and model selection is
itself unstable.